<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/5_Aprendizaje_supervisado/6_Competencia_Seleccion_Mejor_Modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# 🏆 **Tercer parcial: Selección del Mejor Modelo**

**ESTÁ PROHIBIDO EL USO DE GRANDES MODELOS DE LENGUAJE COMO CHATGPT, CLAUDE, GEMINI, ENTRE OTROS, PARA RESOLVER ESTE EJERCICIO**

**Trabajo en grupos de 3**

**Objetivo:** Predecir las ventas de una compañía (`Sales`) teniendo en cuenta su inversión en publicidad.

**Dataset:** `train_df_ventas.csv` disponible en el repositorio del curso.

**IMPORTANTE: Los datos cargados solo corresponden a `train`.**

**Metodología:**
1.  Cargar y explorar los datos.
2.  Preprocesar los datos si es necesario.
3.  De los siguientes modelos, entrenar por lo menos 2:
    - Regresión Lineal
    - Regresión Polinómica
    - KNN Regressor
    - Decision Tree Regressor
    - Random Forest Regressor
    - Gradient Boosting Regressor
    - XGBoost Regressor
4.  Si lo considera necesario, usar `GridSearchCV` con Validación Cruzada (`cv=5`) para optimizar los hiperparámetros. La métrica de optimización debe ser el **RMSE** (Root Mean Squared Error), por lo que debe usar `scoring='neg_root_mean_squared_error'` (el valor será negativo y se multiplicará por -1 al final).
5.  Comparar los modelos y seleccionar el mejor, teniendo en cuenta el menor RMSE.

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma: “Tercer_Parcial_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/Bsy2U83tbc. No olvide indicar claramente cuál es el modelo seleccionado.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Calificación**

La docente, evaluará el modelo seleccionado por ustedes en el `test set`.

El proceso seguirá estas reglas:

- **Criterio de Ganador:** El equipo que tenga todo el procedimiento correcto y obtenga el Root Mean Squared Error (RMSE) más bajo en el test set recibirá una calificación de 5.0.

- **Criterio de Desempate:** En caso de empate en el RMSE, se otorgará la ventaja al equipo que haya entrenado y evaluado más modelos.

- **Escalafón de Notas:** A partir del primer puesto, se restará 0.1 a la nota final por cada posición inferior (2º lugar: 4.9, 3er lugar: 4.8, etc.).

- **Validación de Procedimiento:** Es obligatorio que el código sea reproducible por la docente (no olivde colocar las semillas en los procesos aleatorios). Si el script contiene errores, el equipo quedará fuera de la competencia y se dará una calificación acorde a lo que esté correcto.

**Explicación de las variables:**

- Sales: Ventas (millones USD). --> **Esta es la variable objetivo**
- TV: Gasto en promoción televisiva (millones USD).
- Radio: Gasto en promoción radiofónica (millones USD).
- Social Media: Gasto en promoción en redes sociales (millones USD).
- Influencer: Indica si la promoción se realizó en colaboración con Mega, Macro, Nano o Micro influencers.


**Nombres estudiantes del equipo:**

- Juan Barrantes
- Daniel Caicedo
- Juan Ordoñez

# **Desarrollo**

In [39]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import root_mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
pd.set_option('display.max_columns', None)

In [26]:
from google.colab import drive, files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
path = "/content/drive/MyDrive/Dataset"

In [28]:
os.chdir(path)

In [29]:
df=pd.read_csv("train_df_ventas.csv")
df

,TV,Social Media,Influencer,Radio,Sales
0,17.579874,1.218985,Macro,22.282240,173.612079
1,14.588548,5.478278,Micro,22.097499,140.540703
2,25.183695,2.279885,Mega,22.197715,197.352805
3,12.898275,1.831455,Nano,21.741263,184.047821
4,28.380520,4.338470,Nano,22.607219,318.627485
...,...,...,...,...,...
3631,30.282245,7.438101,Macro,22.467498,283.281154
3632,22.324642,4.001166,Macro,21.825968,266.362757
3633,2.501889,2.170749,Macro,21.824408,69.732518
3634,11.886438,1.558209,Micro,21.253268,82.549072


In [47]:
print("Primeras filas:")
display(df.head())

print("Dimensiones del dataset:")
print(df.shape)

print("Información general:")
print(df.info())

print("Estadísticas descriptivas:")
display(df.describe())

print("Valores faltantes:")
print(df.isnull().sum())

Primeras filas:


,TV,Social Media,Influencer,Radio,Sales
0,17.579874,1.218985,Macro,22.282240,173.612079
1,14.588548,5.478278,Micro,22.097499,140.540703
2,25.183695,2.279885,Mega,22.197715,197.352805
3,12.898275,1.831455,Nano,21.741263,184.047821
4,28.380520,4.338470,Nano,22.607219,318.627485


Dimensiones del dataset:
(3636, 5)
Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3636 entries, 0 to 3635
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   TV            3636 non-null   float64
 1   Social Media  3636 non-null   float64
 2   Influencer    3636 non-null   object 
 3   Radio         3636 non-null   float64
 4   Sales         3636 non-null   float64
dtypes: float64(4), object(1)
memory usage: 142.2+ KB
None
Estadísticas descriptivas:


,TV,Social Media,Radio,Sales
count,3636.000000,3636.000000,3636.000000,3636.000000
mean,18.165917,3.311115,22.085014,192.407780
std,9.671536,2.221452,0.475698,92.553501
min,0.000684,0.000977,21.035102,31.199409
25%,10.598536,1.483576,21.702204,113.173995
50%,17.889588,3.041391,22.045490,189.531695
75%,25.603279,4.804043,22.448079,271.051331
max,48.871161,13.083957,23.224233,364.079751


Valores faltantes:
TV              0
Social Media    0
Influencer      0
Radio           0
Sales           0
dtype: int64


In [31]:
X = df.drop(columns=['Sales'])
y = df['Sales']

In [32]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print("Variables numéricas:", numeric_features)
print("Variables categóricas:", categorical_features)

Variables numéricas: ['TV', 'Social Media', 'Radio']
Variables categóricas: ['Influencer']


In [33]:
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop')

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

#Gradient Boosting

In [36]:
pipe_gbr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])
param_grid_gbr = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [2, 3, 4]
}
grid_gbr = GridSearchCV(
    estimator=pipe_gbr,
    param_grid=param_grid_gbr,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
grid_gbr.fit(X_train, y_train)
print("\nGRADIENT BOOSTING")
print("Mejores parámetros:")
print(grid_gbr.best_params_)

rmse_cv_gbr = -grid_gbr.best_score_
print(f"RMSE CV: {rmse_cv_gbr:.4f}")
y_pred_gbr = grid_gbr.predict(X_test)
rmse_test_gbr = root_mean_squared_error(y_test, y_pred_gbr)

print(f"RMSE Test: {rmse_test_gbr:.4f}")


GRADIENT BOOSTING
Mejores parámetros:
{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100}
RMSE CV: 42.3226
RMSE Test: 42.3980


#XG Boosting


In [38]:
pipe_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        random_state=42,
        objective='reg:squarederror',
        verbosity=0
    ))
])
param_grid_xgb = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [2, 3, 4]
}
grid_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

print("\nXGBOOST")
print("Mejores parámetros:")
print(grid_xgb.best_params_)

rmse_cv_xgb = -grid_xgb.best_score_
print(f"RMSE CV: {rmse_cv_xgb:.4f}")

y_pred_xgb = grid_xgb.predict(X_test)
rmse_test_xgb = root_mean_squared_error(y_test, y_pred_xgb)

print(f"RMSE Test: {rmse_test_xgb:.4f}")


XGBOOST
Mejores parámetros:
{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100}
RMSE CV: 42.3437
RMSE Test: 42.3272


#Linear Regression

In [42]:
pipe_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
param_grid_lr = {}
grid_lr = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
grid_lr.fit(X_train, y_train)

print("\nREGRESIÓN LINEAL")

rmse_cv_lr = -grid_lr.best_score_
print(f"RMSE CV: {rmse_cv_lr:.4f}")
y_pred_lr = grid_lr.predict(X_test)
rmse_test_lr = root_mean_squared_error(y_test, y_pred_lr)

print(f"RMSE Test: {rmse_test_lr:.4f}")


REGRESIÓN LINEAL
RMSE CV: 43.9846
RMSE Test: 44.9918


# Random Forest

In [43]:
pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])
param_grid_rf = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 5, 10],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}
grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
grid_rf.fit(X_train, y_train)
print("\nRANDOM FOREST")
print("Mejores parámetros:")
print(grid_rf.best_params_)

rmse_cv_rf = -grid_rf.best_score_
print(f"RMSE CV: {rmse_cv_rf:.4f}")
y_pred_rf = grid_rf.predict(X_test)
rmse_test_rf = root_mean_squared_error(y_test, y_pred_rf)

print(f"RMSE Test: {rmse_test_rf:.4f}")


RANDOM FOREST
Mejores parámetros:
{'model__max_depth': 5, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 100}
RMSE CV: 42.5228
RMSE Test: 42.6650


#Comparación de resultados

In [46]:
resultados = pd.DataFrame({
    'Modelo': [
        'Regresión Lineal',
        'Random Forest',
        'Gradient Boosting',
        'XGBoost'
    ],
    'RMSE_CV': [
        rmse_cv_lr,
        rmse_cv_rf,
        rmse_cv_gbr,
        rmse_cv_xgb
    ],
    'RMSE_Test': [
        rmse_test_lr,
        rmse_test_rf,
        rmse_test_gbr,
        rmse_test_xgb
    ]
})

resultados = resultados.sort_values(by='RMSE_Test', ascending=True)

print("COMPARACIÓN FINAL")
display(resultados)

COMPARACIÓN FINAL


,Modelo,RMSE_CV,RMSE_Test
3,XGBoost,42.343657,42.327205
2,Gradient Boosting,42.322590,42.398027
1,Random Forest,42.522813,42.665043
0,Regresión Lineal,43.984599,44.991810


#Simulacion del modelo

In [45]:
nueva_observacion = X_test.iloc[[0]]
# Predicción del mejor modelo
prediccion = grid_xgb.predict(nueva_observacion)

print("Ventas predichas:", prediccion[0])
print("Ventas reales:", y_test.iloc[0])

Ventas predichas: 200.72392
Ventas reales: 242.4652608


# **Indica claramente cuál modelo seleccionaste como el mejor**

## Conclusión Final

Pudimos entrenar y evaluar cuatro modelos de regresión:

- Regresión Lineal
- Random Forest
- Gradient Boosting
- XGBoost

## Comparación de Modelos

| Modelo            | RMSE Test |
| ----------------- | --------: |
| XGBoost           |     42.33 |
| Gradient Boosting |     42.40 |
| Random Forest     |     42.67 |
| Regresión Lineal  |     44.99 |




Conclimos que el modelo con mejor desempeño fue "XGBoost", ya que obtuvo el menor error de predicción en el conjunto de prueba (RMSE_Test = 42.33).

El RMSE (Root Mean Squared Error) representa el error promedio que comete el modelo al predecir las ventas, expresado en las mismas unidades de la variable "Sales". Esto significa que, en promedio, las predicciones del modelo XGBoost se desvían aproximadamente en 42.33 unidades respecto a las ventas reales (Como lo vemos en el ultimo ejemplo).

## Interpretación de los Resultados

Nos dimos cuenta de que los modelos basados en árboles (Random Forest, Gradient Boosting y XGBoost) superaran a la Regresión Lineal, esto nos indica que la relación entre las variables de publicidad y las ventas presenta comportamientos no lineales, posibles efectos de saturación, interacciones entre canales publicitarios y umbrales a partir de los cuales la efectividad cambia.

Por ejemplo, aumentar la inversión en publicidad puede generar incrementos importantes en ventas hasta cierto punto, pero después de ese nivel el impacto marginal puede disminuir. Este tipo de patrones es capturado mucho mejor por modelos de ensamble, como nuestro ganador.

XGBoost es el ganador y ya que la diferencia entre el RMSE de validación cruzada "42.34" y el RMSE en test "42.33" es prácticamente nula, lo que demuestra que el modelo generaliza muy bien y no presenta señales de sobreajuste.

## Implicaciones para la Empresa

La empresa ahora cuenta con un modelo para pronosticar ventas a partir de su inversión publicitaria. Esto puede utilizarse para:

* Proyectar ventas futuras.
* Evaluar distintos escenarios de presupuesto.
* Optimizar la asignación de recursos entre campañas.
* Estimar el impacto esperado de cambios en la inversión.

Con un error promedio cercano a 42 unidades, el modelo proporciona una base para apoyar la toma de decisiones comerciales y de marketing.

## Recomendaciones Estratégicas

1. Utilizar XGBoost como modelo oficial de pronóstico

Recomendamos adoptar XGBoost como herramienta principal para estimar ventas esperadas según distintos niveles de inversión que realicen.

2. Realizar simulaciones de presupuesto

El modelo puede utilizarse para responder preguntas como:

* ¿Cuánto aumentarán las ventas si se incrementa la inversión en TV o radio?
* ¿Qué ocurre si se redistribuye el presupuesto entre canales?

3. Identificar variables más influyentes

Podríamos analizar la importancia de variables para determinar qué canales publicitarios tienen mayor impacto.

4. Actualizar el modelo periódicamente

A medida que la empresa recopile nuevos datos, el modelo debe reentrenarse para mantener su precisión y poder seguir generando una prediccion adecuada.

5. Integrar el modelo al proceso de planeación

El modelo puede incorporarse al presupuesto comercial y al diseño de campañas.


## Conclusión

XGBoost fue el modelo con mejor desempeño. Los resultados evidencian que la relación entre inversión publicitaria y ventas es compleja y no lineal, por lo que los modelos avanzados de ensamble son más adecuados que los enfoques lineales tradicionales.

Se recomienda implementar XGBoost como modelo de referencia para apoyar la planificación de marketing, la asignación eficiente del presupuesto publicitario y la proyección de ventas futuras, permitiendo decisiones más informadas y fundamentadas en datos.
